# Chapter 6: Probability Density Functions (PDF)
**Spine:** Think Stats by Allen Downey (adapted for applied analytics work)  
**Libraries:** Production-grade only — `numpy`, `pandas`, `scipy.stats`, `matplotlib`, `seaborn`

---
## What is a PDF?

A PDF describes the relative likelihood of a continuous random variable taking a given value.
Unlike a PMF which gives exact probabilities, a PDF gives **density** — probability only exists over intervals, never at a single point.

**The three relationships you already know:**

$$f(x) = \frac{d}{dx} F(x) \quad \text{(PDF is the derivative of CDF)}$$

$$F(x) = \int_{-\infty}^{x} f(t) \, dt \quad \text{(CDF is the integral of PDF)}$$

$$\int_{-\infty}^{\infty} f(x) \, dx = 1 \quad \text{(PDF must integrate to 1)}$$

In [ ]:
import ssl
import certifi
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.datasets import fetch_california_housing

# Fix SSL certificate verification on Mac
ssl._create_default_https_context = ssl.create_default_context(cafile=certifi.where())

# Consistent style for all plots
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (9, 4)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

# Load California Housing
housing = fetch_california_housing(as_frame=True).frame
income  = housing['MedInc'].values

print('All libraries loaded successfully.')

---
## 1. Why $P(X = x) = 0$ for continuous distributions

This is the most conceptually important thing to understand about continuous distributions.

### The argument from integration

Probability over an interval is defined as:

$$P(a \leq X \leq b) = \int_a^b f(x) \, dx$$

A single point is an interval of width zero. Setting $a = b = x$:

$$P(X = x) = \int_x^x f(x) \, dx = 0$$

An integral over an interval of zero width is always zero — regardless of what $f(x)$ is at that point.

### The intuition

A continuous variable can take infinitely many values. If every single point had positive probability, the total probability would sum to infinity — violating the requirement that total probability equals 1. The only way to distribute probability across infinitely many points while keeping the total at 1 is to assign zero probability to each individual point.

### The practical implication

For continuous data, you never ask *"what is the probability of exactly this value?"*  
You always ask *"what is the probability of falling in this interval?"*

$$P(2.0 \leq \text{income} \leq 3.0) = \int_{2.0}^{3.0} f(x) \, dx \quad \checkmark$$

$$P(\text{income} = 2.5) = 0 \quad \text{always, by definition}$$

### PMF vs PDF — the key distinction

| | PMF | PDF |
|---|---|---|
| Data type | Discrete | Continuous |
| $P(X = x)$ | Can be positive | Always zero |
| Probability over interval | Sum | Integral |
| Values of $f(x)$ | Between 0 and 1 | Can exceed 1 (density, not probability) |

In [ ]:
# Demonstrate P(X = x) = 0 vs P(a <= X <= b) > 0
# Using the fitted log-normal distribution from Chapter 5
params_lognorm = stats.lognorm.fit(income, floc=0)
dist = stats.lognorm(*params_lognorm)

# Probability at an exact point — always 0
x_exact = 3.0
p_exact  = dist.cdf(x_exact) - dist.cdf(x_exact)  # zero width interval

# Probability over intervals of increasing width
intervals = [
    (2.9,  3.1,  'width = 0.2'),
    (2.5,  3.5,  'width = 1.0'),
    (2.0,  4.0,  'width = 2.0'),
]

print(f'P(income = exactly {x_exact}):  {p_exact:.6f}  (always zero)')
print()
print('P(a <= income <= b) over intervals:')
for a, b, label in intervals:
    p = dist.cdf(b) - dist.cdf(a)
    print(f'  [{a}, {b}] ({label}):  {p:.4f}')

print()
print('Key insight: as interval width shrinks toward zero, probability shrinks toward zero.')
print('At exactly zero width, probability is exactly zero.')

In [ ]:
# Visualize — shaded area under PDF represents probability over an interval
x = np.linspace(0, 15, 1000)
pdf_values = dist.pdf(x)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: shade P(2.0 <= X <= 4.0)
a, b = 2.0, 4.0
x_fill = np.linspace(a, b, 500)
axes[0].plot(x, pdf_values, color='steelblue', linewidth=2)
axes[0].fill_between(x_fill, dist.pdf(x_fill), alpha=0.4, color='steelblue',
                     label=f'P({a} ≤ X ≤ {b}) = {dist.cdf(b)-dist.cdf(a):.4f}')
axes[0].set_title('Probability = area under PDF over an interval')
axes[0].set_xlabel('Median income ($10,000s)')
axes[0].set_ylabel('Density')
axes[0].legend()

# Right: show that PDF value can exceed 1 — density is not probability
# Use a narrow normal distribution where peak exceeds 1
narrow_dist = stats.norm(loc=0, scale=0.1)
x_narrow = np.linspace(-0.5, 0.5, 1000)
axes[1].plot(x_narrow, narrow_dist.pdf(x_narrow), color='darkorange', linewidth=2)
axes[1].axhline(y=1, color='crimson', linestyle='--', linewidth=1, label='y = 1')
axes[1].set_title('PDF values can exceed 1 — density ≠ probability')
axes[1].set_xlabel('x')
axes[1].set_ylabel('Density f(x)')
axes[1].legend()

plt.tight_layout()
plt.savefig('pdf_concepts.png', dpi=150, bbox_inches='tight')
plt.show()

print('Right plot: f(x) peaks well above 1 — valid because it is density, not probability.')
print('The area under the entire curve still integrates to exactly 1.')

---
## 2. Kernel Density Estimation (KDE)

### The problem KDE solves

In Chapter 5 we fitted named theoretical distributions — log-normal, normal, exponential.
That approach requires a strong assumption: *"I believe my data follows this specific distribution."*

KDE makes no such assumption. It estimates the PDF directly from the data — letting the data speak for itself without forcing it into a predefined shape.

### How KDE works

For each observation $x_i$ in your data, place a small smooth curve — called a **kernel** — centered at that point. Sum all the kernels together and normalize so the result integrates to 1. The result is a smooth estimate of the underlying PDF.

$$\hat{f}(x) = \frac{1}{nh} \sum_{i=1}^{n} K\left(\frac{x - x_i}{h}\right)$$

Where:
- $n$ = number of observations
- $h$ = **bandwidth** — controls how smooth the estimate is
- $K$ = **kernel function** — the shape of each individual bump (usually Gaussian)

### The bandwidth parameter

Bandwidth $h$ is the most important parameter in KDE:

- **Too small** — curve is jagged and noisy, overfits the data
- **Too large** — curve is oversmoothed, loses important features
- **Just right** — smooth curve that captures the true shape

In production, `scipy` and `seaborn` estimate bandwidth automatically using **Silverman's rule of thumb**:

$$h = 0.9 \cdot \min\left(\sigma, \frac{IQR}{1.34}\right) \cdot n^{-1/5}$$

In [ ]:
# KDE using scipy.stats.gaussian_kde — production tool
kde = stats.gaussian_kde(income)

x = np.linspace(income.min(), income.max(), 1000)
kde_values = kde(x)

# Fitted log-normal PDF for comparison
lognorm_pdf = stats.lognorm.pdf(x, *params_lognorm)

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(income, bins=50, density=True, color='steelblue', alpha=0.4, label='Empirical')
ax.plot(x, kde_values,   color='steelblue',  linewidth=2, label='KDE (no distribution assumption)')
ax.plot(x, lognorm_pdf,  color='crimson',    linewidth=2, linestyle='--', label='Fitted log-normal PDF')
ax.set_title('KDE vs fitted log-normal PDF — median income')
ax.set_xlabel('Median income ($10,000s)')
ax.set_ylabel('Density')
ax.legend()

plt.tight_layout()
plt.savefig('kde_vs_lognormal.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'KDE bandwidth (Silverman): {kde.factor:.4f}')
print()
print('Where KDE and log-normal agree: the log-normal assumption is defensible.')
print('Where they diverge: KDE is revealing structure the log-normal model misses.')

In [ ]:
# Demonstrate effect of bandwidth on KDE shape
# This is the most important tuning parameter to understand
bandwidths = [0.05, 0.2, 0.5, 2.0]
labels     = ['Too narrow (undersmoothed)', 'Narrow', 'Default (Silverman)', 'Too wide (oversmoothed)']
colors     = ['crimson', 'darkorange', 'steelblue', 'seagreen']

fig, ax = plt.subplots(figsize=(11, 5))
ax.hist(income, bins=50, density=True, color='lightgray', alpha=0.5, label='Empirical')

for bw, label, color in zip(bandwidths, labels, colors):
    kde_bw = stats.gaussian_kde(income, bw_method=bw)
    ax.plot(x, kde_bw(x), color=color, linewidth=2, label=f'h={bw} — {label}')

ax.set_title('Effect of bandwidth on KDE')
ax.set_xlabel('Median income ($10,000s)')
ax.set_ylabel('Density')
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('kde_bandwidth.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 3. KDE for comparing distributions
**Real data:** California Housing — high vs low income neighborhoods  

KDE is particularly powerful for overlapping distribution comparisons — cleaner than overlapping histograms because there are no bin-width artifacts.

In [ ]:
high_income_val = housing[housing['MedInc'] >= 5.0]['MedHouseVal'].values
low_income_val  = housing[housing['MedInc'] <  3.0]['MedHouseVal'].values

kde_high = stats.gaussian_kde(high_income_val)
kde_low  = stats.gaussian_kde(low_income_val)

x_range = np.linspace(
    min(high_income_val.min(), low_income_val.min()),
    max(high_income_val.max(), low_income_val.max()),
    1000
)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: overlapping histograms — messy
axes[0].hist(high_income_val, bins=40, density=True, alpha=0.5, color='steelblue',  label='High income')
axes[0].hist(low_income_val,  bins=40, density=True, alpha=0.5, color='darkorange', label='Low income')
axes[0].set_title('Overlapping histograms — bin artifacts visible')
axes[0].set_xlabel('Median house value ($100,000s)')
axes[0].set_ylabel('Density')
axes[0].legend()

# Right: KDE comparison — cleaner
axes[1].plot(x_range, kde_high(x_range), color='steelblue',  linewidth=2, label='High income (MedInc ≥ 5.0)')
axes[1].plot(x_range, kde_low(x_range),  color='darkorange', linewidth=2, label='Low income  (MedInc < 3.0)')
axes[1].fill_between(x_range, kde_high(x_range), alpha=0.15, color='steelblue')
axes[1].fill_between(x_range, kde_low(x_range),  alpha=0.15, color='darkorange')
axes[1].set_title('KDE comparison — no bin artifacts')
axes[1].set_xlabel('Median house value ($100,000s)')
axes[1].set_ylabel('Density')
axes[1].legend()

plt.tight_layout()
plt.savefig('kde_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print('KDE removes the subjectivity of bin width choice.')
print('The shape of the comparison is determined by the data, not by how you bin it.')

---
## 4. KDE utility functions — reusable in production

In [ ]:
def kde_pdf(data: np.ndarray, x: np.ndarray, bw_method=None) -> np.ndarray:
    """
    Estimate PDF at points x using KDE.
    Production-safe: uses only scipy.stats.gaussian_kde.

    Parameters
    ----------
    data      : raw observations
    x         : points at which to evaluate the PDF
    bw_method : bandwidth — None uses Silverman's rule automatically
    """
    kde = stats.gaussian_kde(data, bw_method=bw_method)
    return kde(x)


def kde_prob_between(data: np.ndarray, a: float, b: float) -> float:
    """
    Estimate P(a <= X <= b) from KDE using numerical integration.
    Production-safe: uses only scipy.
    """
    from scipy.integrate import quad
    kde = stats.gaussian_kde(data)
    prob, _ = quad(kde, a, b)
    return prob


def plot_kde(data: np.ndarray, title: str = 'KDE', ax=None, **kwargs):
    """
    Reusable production-safe KDE plotter.
    """
    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 4))
    kde  = stats.gaussian_kde(data)
    x    = np.linspace(data.min(), data.max(), 1000)
    ax.plot(x, kde(x), **kwargs)
    ax.set_title(title)
    ax.set_ylabel('Density')
    return ax


# Demonstrate
print('--- KDE analytics on median income ---')
print(f'P(2.0 <= income <= 3.0):   {kde_prob_between(income, 2.0, 3.0):.4f}')
print(f'P(income >= 8.0):           {kde_prob_between(income, 8.0, income.max()):.4f}')
print(f'P(income <= 2.0):           {kde_prob_between(income, income.min(), 2.0):.4f}')

---
## 5. KDE vs parametric fitting — when to use which

| | Parametric (Chapter 5) | KDE (this chapter) |
|---|---|---|
| Assumption | Data follows a named distribution | No distributional assumption |
| Parameters | Few (e.g. $\mu$, $\sigma$) | One — bandwidth $h$ |
| Extrapolation | Good — theoretical tails extend naturally | Poor — unreliable beyond observed data range |
| Interpretability | High — parameters have meaning | Low — no simple summary |
| Flexibility | Low — constrained to distribution shape | High — adapts to any shape |
| Best when | You have domain knowledge about the distribution | You do not, or the data has unusual shape |

**In production the typical workflow is:**
1. Use KDE first — let the data reveal its shape without assumptions
2. Use that shape to hypothesize a named distribution
3. Fit the named distribution and validate with KS test and Q-Q plot
4. Use the named distribution for inference and extrapolation

KDE is the exploration tool. Parametric fitting is the production model.

---
## Summary

**Why $P(X = x) = 0$:**  
An integral over an interval of zero width is always zero. Probability only exists over intervals for continuous distributions — never at a single point.

**KDE in one sentence:**  
Place a small Gaussian kernel at each observation, sum them all, normalize — the result is a smooth non-parametric estimate of the underlying PDF.

**Key parameter:** Bandwidth $h$ — controls smoothness. Use Silverman's rule (default) in production unless you have a specific reason to override it.

**Production libraries used:**
- `scipy.stats.gaussian_kde` — KDE estimation and evaluation
- `scipy.integrate.quad` — numerical integration for probability over intervals
- `scipy.stats` — parametric PDFs for comparison
- `matplotlib` / `seaborn` — visualization

**Next:** Chapter 7 — Relationships between variables